# Sentiment Classification for Hausa: A Low-Resource Language Study

This notebook rebuilds the Hausa AfriSenti sentiment project in a clean, notebook-first form.
It re-uses the already-trained models and the experiment registry rather than re-running every experiment from scratch.

**Task.** Classify Hausa social-media text as positive, neutral or negative.

**Headline result.** A classical word + character TF-IDF model with a calibrated LinearSVC reaches **test macro-F1 0.782**, outperforming the best recurrent model (0.746) and a budget-constrained LoRA transformer (0.590).

**Data.** Hausa portion of AfriSenti-SemEval 2023 Subtask A (train 14 172, dev 2 677, test 5 303).


## 1. Setup and data loading

### 1.1 Reproducibility and imports


In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
np.random.seed(SEED)

ROOT = Path("..") if Path("..").joinpath("results").exists() else Path(".")
DATA = ROOT / "data" / "processed"
MODELS = ROOT / "models"
RESULTS = ROOT / "results"

print("ROOT =", ROOT.resolve())
print("DATA exists:", DATA.exists())
print("MODELS exists:", MODELS.exists())
print("RESULTS exists:", RESULTS.exists())


### 1.2 Load the experiment registry

All previously recorded runs live in `results/experiments.json`.  
This is the single source of truth for the numbers reported later.


In [ ]:
with open(RESULTS / "experiments.json", encoding="utf-8") as f:
    registry = json.load(f)

runs = registry["runs"]
print(f"Loaded {len(runs)} experiment runs:")
for name in runs:
    test = runs[name]["splits"].get("test", {})
    f1 = test.get("macro_f1")
    print(f"  {name:55s}  test macro-F1 = {f1}")


### 1.3 Load the training split

Only the training CSV is present in this clean package so far.  
Dev / dev_noleak / test can be added later; the registry already contains the evaluation numbers.


In [ ]:
train = pd.read_csv(DATA / "afrisenti_hau_train.csv")
print("Train shape:", train.shape)
print("Columns:", list(train.columns))
print()
print(train[["label", "label_id"]].value_counts().sort_index())
print()
train.head(3)


#### 1.3 Interpretation

- 14 172 training examples.
- Labels are balanced (positive / neutral / negative).
- Extra columns (`text_bow`, emoji flags, code-switch flag, \ldots) come from the original preprocessing pipeline and are kept for later analysis cells.


## 2. Headline results from the registry

We do not re-train every model here. Instead we read the already-recorded test scores and build the comparison table that the report will use.


In [ ]:
def test_f1(name):
    return runs[name]["splits"]["test"]["macro_f1"]

headline = pd.DataFrame([
    {"family": "Majority baseline", "run": "B0 majority class",
     "model": "DummyClassifier (most frequent)", "test_macro_f1": test_f1("B0 majority class")},
    {"family": "Classical", "run": "B4 word+char TF-IDF + LinearSVC",
     "model": "Word + char TF-IDF + calibrated LinearSVC", "test_macro_f1": test_f1("B4 word+char TF-IDF + LinearSVC")},
    {"family": "Classical (ablation)", "run": "B3 char TF-IDF + LogReg",
     "model": "Character TF-IDF + LogReg", "test_macro_f1": test_f1("B3 char TF-IDF + LogReg")},
    {"family": "Recurrent", "run": "R7 BiLSTM attention pooling",
     "model": "BiLSTM + attention (Word2Vec mixed, fine-tuned)", "test_macro_f1": test_f1("R7 BiLSTM attention pooling")},
    {"family": "Transformer", "run": "T1 afro-xlmr-mini LoRA (CPU)",
     "model": "AfroXLMR-mini LoRA (1 epoch, CPU)", "test_macro_f1": test_f1("T1 afro-xlmr-mini LoRA (CPU)")},
])

headline = headline.sort_values("test_macro_f1", ascending=False).reset_index(drop=True)
headline


#### 2 Interpretation

- The classical word+character model is clearly strongest (0.782).
- Character n-grams alone already reach 0.775, showing that sub-word features matter for Hausa social-media text (spelling variation, missing diacritics, code-switching).
- The best recurrent model (BiLSTM + attention) lands at 0.746 — competitive but still below the classical baseline.
- The transformer run is a **budget-constrained** CPU LoRA experiment (1 epoch). It should not be read as evidence that transformers are inherently worse; it simply did not converge fully under the submission constraints.


## 3. Classical model family

All classical runs share the same preprocessing (`text_bow` = cleaned text with `@user` / `<url>` stripped for bag-of-words features).


In [ ]:
classical_names = [n for n in runs if n.startswith("B") and "DS" not in n]
rows = []
for name in classical_names:
    s = runs[name]["splits"]["test"]
    rows.append({
        "run": name,
        "test_macro_f1": s["macro_f1"],
        "test_acc": s["accuracy"],
        "pos_f1": s["per_class"]["positive"]["f1"],
        "neu_f1": s["per_class"]["neutral"]["f1"],
        "neg_f1": s["per_class"]["negative"]["f1"],
    })
classical_table = pd.DataFrame(rows).sort_values("test_macro_f1", ascending=False)
classical_table


#### 3 Interpretation

- Adding character n-grams on top of word n-grams is the single biggest gain.
- LinearSVC and Logistic Regression are nearly tied; the calibrated LinearSVC was kept as the deployment model.
- ComplementNB is competitive but clearly weaker.


## 4. Recurrent model family

All R-series models use Word2Vec embeddings (in-domain or mixed corpus) and bidirectional RNNs.


In [ ]:
rnn_names = [n for n in runs if n.startswith("R")]
rows = []
for name in rnn_names:
    s = runs[name]["splits"]["test"]
    cfg = runs[name].get("config", {})
    rows.append({
        "run": name,
        "test_macro_f1": s["macro_f1"],
        "rnn": cfg.get("rnn_type", ""),
        "pooling": cfg.get("pooling", ""),
        "emb": cfg.get("emb", ""),
        "freeze_emb": cfg.get("freeze_emb", ""),
    })
rnn_table = pd.DataFrame(rows).sort_values("test_macro_f1", ascending=False)
rnn_table


#### 4 Interpretation

- R7 (BiLSTM + additive attention, mixed Word2Vec, fine-tuned) is the best recurrent run at 0.746.
- Attention pooling helps over simple mean/max pooling.
- None of the recurrent configurations surpass the classical word+char baseline.


## 5. Transformer (budget-constrained)

Only the CPU LoRA run of `Davlan/afro-xlmr-mini` is present in this package.


In [ ]:
t1 = runs["T1 afro-xlmr-mini LoRA (CPU)"]
print("Config:")
for k, v in t1["config"].items():
    print(f"  {k}: {v}")
print()
print("Test metrics:")
for k, v in t1["splits"]["test"].items():
    if k != "per_class":
        print(f"  {k}: {v}")
print()
print("Per-class F1:")
for lab, d in t1["splits"]["test"]["per_class"].items():
    print(f"  {lab}: {d['f1']:.4f}")


#### 5 Interpretation

- 1 epoch on CPU with LoRA (r=8, ~0.25 % of parameters trainable).
- Macro-F1 0.590 is far below the classical and recurrent models.
- This is expected under the compute budget; a longer GPU fine-tune (or full fine-tuning of a larger African-language model) would be needed for a fair transformer comparison.


## 6. Data-integrity notes (from the original project)

These findings were established in the original research pipeline and are recorded here so the clean package stays self-contained.

1. **Label mapping.** Official GitHub and Hugging Face versions agree: `0 = positive`, `1 = neutral`, `2 = negative`.
2. **Leakage.** 215 development examples are exact duplicates of training rows. A `dev_noleak` split (2 462 rows) removes them and was used for model selection.
3. **Surface mismatch.** Train/dev are almost entirely emoji-bearing; the released test set has almost no emoji and was cleaned differently (mentions, links, etc.).
4. **Conflicting labels.** 13 texts appear with more than one label across the corpus.
5. **Ordering.** Official files are ordered in contiguous label blocks rather than randomly shuffled.

These issues are the reason the original project distinguishes `dev` vs `dev_noleak` and why domain-shift probes were run.


## 7. Model artefacts available in this package


In [ ]:
print("Models directory contents:")
for p in sorted(MODELS.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f"  {p.relative_to(MODELS)}  ({size_mb:.2f} MB)")


### 7.1 Quick load check of the classical model

The classical model is a scikit-learn pipeline (vectorizer + calibrated LinearSVC) saved with joblib.


In [ ]:
import joblib

clf = joblib.load(MODELS / "classical_best.pkl")
print(type(clf))
print(clf)

# smoke prediction
sample = train["text_bow"].iloc[0]
print("\nSample text:", sample[:120], "...")
if hasattr(clf, "predict_proba"):
    proba = clf.predict_proba([sample])[0]
    print("Probabilities:", dict(zip(["positive", "neutral", "negative"], proba.round(4))))
else:
    pred = clf.predict([sample])[0]
    print("Predicted label id:", pred)


#### 7.1 Interpretation

The classical model loads and produces a prediction.  
This is the model that will be served as the primary classifier in the Gradio app.


## 8. Placeholders for later steps

The following sections will be filled once the remaining split files (dev / test) or further analysis artefacts are added:

- **8.1** Load dev_noleak and test CSVs; recompute class balance and surface statistics.
- **8.2** Domain-shift probe (emoji / mention stripping) using the numbers already in the registry under the `DS C1` / `DS C2` runs.
- **8.3** Error taxonomy (code-switching, negation, short tweets, high-OOV, \ldots).
- **8.4** Out-of-domain / cross-lingual notes.
- **8.5** Export the two models the app will use (classical + optional second opinion).

Until then, the registry and the three model artefacts already support the headline claims of the report and a minimal working demo.


## 9. Summary

| Claim | Evidence |
|-------|----------|
| Classical word+char TF-IDF is strongest | B4 test macro-F1 = 0.782 |
| Character features matter | B3 (char only) = 0.775 vs B1 (word only) = 0.739 |
| Recurrent models do not beat classical | Best R7 = 0.746 |
| Transformer under-trained on CPU | T1 = 0.590 (1 epoch LoRA) |
| Data has leakage and surface mismatch | Documented in original integrity report; reflected in registry splits |

Next concrete step after this notebook: write the short report (`report/Hausa_Sentiment_Report.md`) that re-uses these numbers, then the Gradio app.
